# Demo: LLM Judge for Clarifying-Question Uncertainty

This notebook uses the `llm_judge` framework to classify Phase 1 clarifying questions as `ALEATORIC` or `EPISTEMIC`, with few-shot guidance, a smoke test, and a validity check before full CSV classification.

## 1) Setup and Config

In [5]:
import csv
import logging
import os
import re
import time
from pathlib import Path

import pandas as pd
from tenacity import retry, retry_if_exception_type, stop_after_attempt, wait_exponential

from llm_judge import (
    LLMProvider,
    LLMProviderError,
    LLMJudge,
    FewShotExample,
)

# ===== Config =====
MODEL_ID = "gemini-2.5-flash"
API_VERSIONS_TO_TRY = ["v1beta", "v1"]
FALLBACK_MODELS = ["gemini-2.0-flash"]
MAX_OUTPUT_TOKENS = 256
TEMPERATURE = 0.0
REQUEST_INTERVAL = 1.0

INSTRUCTIONS_PATH = Path("instructions.txt")
PHASE1_RESULTS_CANDIDATES = [
    Path("../Phase 1/outputs/phase1_results.csv"),
    Path("../Phase 1/phase1_results.csv"),
    Path("phase1_results.csv"),
]
OUTPUT_PATH = Path("phase1_cq_type_classified.csv")

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-8s | %(name)s | %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
)
logger = logging.getLogger("judge_demo")

def load_dotenv(path: Path = Path("../Phase 1/.env")):
    if not path.exists():
        path = Path(".env")
    if not path.exists():
        raise FileNotFoundError("No .env file found in ../Phase 1/.env or ./.env")
    with path.open("r", encoding="utf-8") as f:
        for raw_line in f:
            line = raw_line.strip()
            if not line or line.startswith("#") or "=" not in line:
                continue
            key, value = line.split("=", 1)
            os.environ.setdefault(key.strip(), value.strip())
    return path

env_path_used = load_dotenv()
if "VERTEX_API_KEY" not in os.environ:
    raise RuntimeError("VERTEX_API_KEY not found after loading .env")

print(f"Loaded .env from: {env_path_used}")
print(f"Instructions path: {INSTRUCTIONS_PATH.resolve()}")
print(f"Output path: {OUTPUT_PATH.resolve()}")

Loaded .env from: ..\Phase 1\.env
Instructions path: D:\final_project\pilot_study_2\LLM Judge\instructions.txt
Output path: D:\final_project\pilot_study_2\LLM Judge\phase1_cq_type_classified.csv


## 2) Few-Shot Examples and Label Parser

In [6]:
FEW_SHOT_EXAMPLES: list[FewShotExample] = [
    FewShotExample(
        input="When you say you've been feeling off lately, can you describe what that feels like?",
        expected_output="ALEATORIC",
        explanation=(
            "The phrase 'feeling off' is inherently ambiguous — it could mean "
            "fatigue, dizziness, low mood, nausea, or many other things. "
            "The question tries to narrow the meaning but the patient's description "
            "could still resolve into multiple valid interpretations."
        ),
    ),
    FewShotExample(
        input="What do you mean when you say the pain is 'bad' — is it sharp, dull, burning, or pressure-like?",
        expected_output="ALEATORIC",
        explanation=(
            "The word 'bad' carries no clinical specificity. The LLM is asking "
            "because the same word could describe completely different pain "
            "characters pointing to different diagnoses — the ambiguity is in "
            "the description itself, not a missing fact."
        ),
    ),
    FewShotExample(
        input="You mentioned discomfort in your chest — do you mean pain, tightness, shortness of breath, or something else?",
        expected_output="ALEATORIC",
        explanation=(
            "'Chest discomfort' is vague enough to encompass cardiac, respiratory, "
            "musculoskeletal, or gastrointestinal causes. The LLM cannot assign "
            "a single meaning without the patient interpreting their own experience, "
            "which may still leave multiple possibilities open."
        ),
    ),
    FewShotExample(
        input="Do you have your most recent chest X-ray results available?",
        expected_output="EPISTEMIC",
        explanation=(
            "The LLM understands the clinical picture but needs a concrete "
            "diagnostic result to reason further. The X-ray either exists or "
            "it doesn't — providing it fully closes the knowledge gap."
        ),
    ),
    FewShotExample(
        input="What is your current temperature reading?",
        expected_output="EPISTEMIC",
        explanation=(
            "A specific measurable fact. The LLM is not confused about what "
            "the patient means — it simply needs the value to assess whether "
            "fever is present. The answer resolves the uncertainty completely."
        ),
    ),
    FewShotExample(
        input="Have you been diagnosed with hypertension before?",
        expected_output="EPISTEMIC",
        explanation=(
            "A historical medical fact with a definitive yes or no answer. "
            "The LLM is filling a knowledge gap about the patient's background "
            "— the answer exists and fully resolves what it needs to know."
        ),
    ),
]

VALID_LABELS = {"ALEATORIC", "EPISTEMIC"}

def robust_label_parser(raw_text: str) -> str:
    if not raw_text:
        return "ERROR"
    txt = raw_text.strip().upper()

    exact = txt.strip("` 	\n\r\"'")
    if exact in VALID_LABELS:
        return exact

    m = re.search(r"\b(ALEATORIC|EPISTEMIC)\b", txt)
    if m:
        return m.group(1)

    return "INVALID_LABEL"

print(f"Few-shot examples loaded: {len(FEW_SHOT_EXAMPLES)}")

Few-shot examples loaded: 6


## 3) Gemini Provider Adapter for LLM Judge

In [7]:
from google import genai
from google.genai import types

class GeminiJudgeProvider(LLMProvider):
    def __init__(
        self,
        api_key: str,
        model: str = MODEL_ID,
        api_versions: list[str] | None = None,
        fallback_models: list[str] | None = None,
        max_output_tokens: int = MAX_OUTPUT_TOKENS,
        temperature: float = TEMPERATURE,
    ) -> None:
        self._model = model
        self._api_versions = api_versions or ["v1beta", "v1"]
        self._fallback_models = fallback_models or []
        self._max_output_tokens = max_output_tokens
        self._temperature = temperature
        self._api_key = api_key

        self._clients: dict[str, genai.Client] = {}
        for ver in self._api_versions:
            self._clients[ver] = genai.Client(
                api_key=self._api_key,
                http_options=types.HttpOptions(api_version=ver),
            )

    @property
    def provider_name(self) -> str:
        return "GoogleGenAI"

    @property
    def model_name(self) -> str:
        return self._model

    @retry(
        retry=retry_if_exception_type(Exception),
        wait=wait_exponential(multiplier=1, min=2, max=30),
        stop=stop_after_attempt(5),
        reraise=True,
    )
    def generate_response(self, prompt: str) -> str:
        candidates = [self._model] + [m for m in self._fallback_models if m != self._model]
        last_exc = None

        for ver in self._api_versions:
            client = self._clients[ver]
            for model in candidates:
                try:
                    resp = client.models.generate_content(
                        model=model,
                        contents=prompt,
                        config=types.GenerateContentConfig(
                            temperature=self._temperature,
                            max_output_tokens=self._max_output_tokens,
                        ),
                    )
                    text = getattr(resp, "text", None)
                    if text:
                        return text

                    parts = []
                    for part in resp.candidates[0].content.parts:
                        part_text = getattr(part, "text", "")
                        if part_text:
                            parts.append(part_text)
                    joined = "\n".join(parts).strip()
                    if joined:
                        return joined

                    raise LLMProviderError("Empty response text from Gemini")
                except Exception as exc:
                    last_exc = exc
                    logger.warning(
                        "Gemini provider attempt failed | model=%s api_version=%s | %s",
                        model,
                        ver,
                        exc,
                    )

        raise LLMProviderError(f"All Gemini attempts failed. Last error: {last_exc}")

provider = GeminiJudgeProvider(
    api_key=os.environ["VERTEX_API_KEY"],
    model=MODEL_ID,
    api_versions=API_VERSIONS_TO_TRY,
    fallback_models=FALLBACK_MODELS,
    max_output_tokens=MAX_OUTPUT_TOKENS,
    temperature=TEMPERATURE,
)

judge = LLMJudge(
    provider=provider,
    instructions_path=INSTRUCTIONS_PATH,
    few_shot_examples=FEW_SHOT_EXAMPLES,
    label_parser=robust_label_parser,
)

print(f"Judge ready | provider={provider.provider_name} | model={provider.model_name}")

2026-04-27 15:09:46 | INFO     | llm_judge | LLMJudge initialised | provider=GoogleGenAI | model=gemini-2.5-flash | few_shot_count=6
2026-04-27 15:09:46 | INFO     | llm_judge | Loading instructions from 'instructions.txt'
2026-04-27 15:09:46 | INFO     | llm_judge | Instructions loaded successfully | chars=1252


Judge ready | provider=GoogleGenAI | model=gemini-2.5-flash


## 4) Smoke Test

In [8]:
smoke_question = "What is your current temperature reading?"
smoke_result = judge.evaluate(smoke_question)

print("Smoke test input:", smoke_question)
print("Smoke test label:", smoke_result.label)
print("Smoke test raw response:", smoke_result.raw_response)
print("Smoke test error:", smoke_result.error)

if smoke_result.error:
    raise RuntimeError(f"Smoke test provider error: {smoke_result.error}")
if smoke_result.label not in VALID_LABELS:
    raise RuntimeError(
        f"Smoke test returned invalid label: {smoke_result.label}. Raw: {smoke_result.raw_response}"
    )

print("Smoke test passed.")

2026-04-27 15:09:48 | INFO     | llm_judge | Evaluating text | provider=GoogleGenAI | text_preview='What is your current temperature reading?...'
2026-04-27 15:09:48 | INFO     | google_genai.models | AFC is enabled with max remote calls: 10.
2026-04-27 15:09:50 | INFO     | httpx | HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent "HTTP/1.1 200 OK"
2026-04-27 15:09:50 | INFO     | llm_judge | Evaluation complete | label='EPISTEMIC' | latency=1.984s


Smoke test input: What is your current temperature reading?
Smoke test label: EPISTEMIC
Smoke test raw response: EPISTEMIC
Smoke test error: None
Smoke test passed.


## 5) Validity Check on Few-Shot Anchors

In [9]:
validity_rows = []
for ex in FEW_SHOT_EXAMPLES:
    res = judge.evaluate(ex.input)
    validity_rows.append({
        "input": ex.input,
        "expected": ex.expected_output,
        "predicted": res.label,
        "correct": res.label == ex.expected_output,
        "error": res.error,
        "raw_response": res.raw_response,
    })
    time.sleep(REQUEST_INTERVAL)

validity_df = pd.DataFrame(validity_rows)
accuracy = validity_df["correct"].mean() if len(validity_df) else 0.0
invalid_count = (~validity_df["predicted"].isin(list(VALID_LABELS))).sum()
error_count = validity_df["error"].notna().sum()

print(f"Few-shot validity accuracy: {accuracy:.2%}")
print(f"Invalid labels: {invalid_count}")
print(f"Provider errors: {error_count}")
display(validity_df[["expected", "predicted", "correct", "error", "input"]])

if invalid_count > 0:
    raise RuntimeError("Validity check failed: invalid labels found.")

2026-04-27 15:09:59 | INFO     | llm_judge | Evaluating text | provider=GoogleGenAI | text_preview='When you say you've been feeling off lately, can you describ...'
2026-04-27 15:09:59 | INFO     | google_genai.models | AFC is enabled with max remote calls: 10.
2026-04-27 15:10:01 | INFO     | httpx | HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent "HTTP/1.1 200 OK"
2026-04-27 15:10:01 | INFO     | llm_judge | Evaluation complete | label='ALEATORIC' | latency=2.158s
2026-04-27 15:10:02 | INFO     | llm_judge | Evaluating text | provider=GoogleGenAI | text_preview='What do you mean when you say the pain is 'bad' — is it shar...'
2026-04-27 15:10:02 | INFO     | google_genai.models | AFC is enabled with max remote calls: 10.
2026-04-27 15:10:04 | INFO     | httpx | HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent "HTTP/1.1 200 OK"
2026-04-27 15:10:04 | INFO     | llm_jud

Few-shot validity accuracy: 100.00%
Invalid labels: 0
Provider errors: 0


,expected,predicted,correct,error,input
0,ALEATORIC,ALEATORIC,True,None,"When you say you've been feeling off lately, c..."
1,ALEATORIC,ALEATORIC,True,None,What do you mean when you say the pain is 'bad...
2,ALEATORIC,ALEATORIC,True,None,You mentioned discomfort in your chest — do yo...
3,EPISTEMIC,EPISTEMIC,True,None,Do you have your most recent chest X-ray resul...
4,EPISTEMIC,EPISTEMIC,True,None,What is your current temperature reading?
5,EPISTEMIC,EPISTEMIC,True,None,Have you been diagnosed with hypertension before?


## 6) Load Phase 1 Results

In [10]:
phase1_path = None
for p in PHASE1_RESULTS_CANDIDATES:
    if p.exists():
        phase1_path = p
        break

if phase1_path is None:
    raise FileNotFoundError(
        "Could not find phase1_results.csv. Checked: " + ", ".join(str(p) for p in PHASE1_RESULTS_CANDIDATES)
    )

phase1_df = pd.read_csv(phase1_path)
if "clarifying_question" not in phase1_df.columns:
    raise ValueError("Column 'clarifying_question' not found in phase1 results CSV.")
if "id" not in phase1_df.columns:
    phase1_df["id"] = [f"row_{i}" for i in range(len(phase1_df))]

work_df = phase1_df.copy()
work_df["clarifying_question"] = work_df["clarifying_question"].fillna("").astype(str).str.strip()
work_df = work_df[work_df["clarifying_question"] != ""].copy()
work_df = work_df[~work_df["clarifying_question"].str.upper().eq("BLOCKED")].copy()

print(f"Using phase1 results: {phase1_path.resolve()}")
print(f"Total rows in source CSV: {len(phase1_df)}")
print(f"Rows with valid clarifying_question: {len(work_df)}")
display(work_df[["id", "clarifying_question"]].head(5))

Using phase1 results: D:\final_project\pilot_study_2\Phase 1\outputs\phase1_results.csv
Total rows in source CSV: 7
Rows with valid clarifying_question: 7


,id,clarifying_question
0,medqa_3,"Are you experiencing any fever, chills, or spe..."
1,medqa_1,"Are the blisters itchy, painful, or burning?"
2,medqa_7,Does he experience shortness of breath when ly...
3,medqa_4,Has your difficulty breathing worsened when ly...
4,medqa_11,Has she experienced any uterine contractions o...


## 7) Classify Clarifying Questions (Full Run)

In [11]:
rows_out = []
for i, row in enumerate(work_df.to_dict(orient="records"), start=1):
    q = row["clarifying_question"]
    result = judge.evaluate(q)

    row_out = dict(row)
    row_out["cq_type"] = result.label
    row_out["judge_raw_response"] = result.raw_response
    row_out["judge_provider"] = result.provider
    row_out["judge_model"] = result.model
    row_out["judge_latency_seconds"] = result.latency_seconds
    row_out["judge_error"] = result.error
    row_out["judge_timestamp"] = result.timestamp
    rows_out.append(row_out)

    print(f"[{i}/{len(work_df)}] id={row_out.get('id')} label={result.label} error={result.error}")
    time.sleep(REQUEST_INTERVAL)

classified_df = pd.DataFrame(rows_out)
classified_df.to_csv(OUTPUT_PATH, index=False)

print(f"Saved classified results to: {OUTPUT_PATH.resolve()}")
print(f"Rows saved: {len(classified_df)}")

2026-04-27 15:11:07 | INFO     | llm_judge | Evaluating text | provider=GoogleGenAI | text_preview='Are you experiencing any fever, chills, or specific muscle p...'
2026-04-27 15:11:07 | INFO     | google_genai.models | AFC is enabled with max remote calls: 10.
2026-04-27 15:11:10 | INFO     | httpx | HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent "HTTP/1.1 200 OK"
2026-04-27 15:11:10 | INFO     | llm_judge | Evaluation complete | label='EPISTEMIC' | latency=2.997s


[1/7] id=medqa_3 label=EPISTEMIC error=None


2026-04-27 15:11:11 | INFO     | llm_judge | Evaluating text | provider=GoogleGenAI | text_preview='Are the blisters itchy, painful, or burning?...'
2026-04-27 15:11:11 | INFO     | google_genai.models | AFC is enabled with max remote calls: 10.
2026-04-27 15:11:14 | INFO     | httpx | HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent "HTTP/1.1 200 OK"
2026-04-27 15:11:14 | INFO     | llm_judge | Evaluation complete | label='EPISTEMIC' | latency=3.499s


[2/7] id=medqa_1 label=EPISTEMIC error=None


2026-04-27 15:11:15 | INFO     | llm_judge | Evaluating text | provider=GoogleGenAI | text_preview='Does he experience shortness of breath when lying flat or do...'
2026-04-27 15:11:15 | INFO     | google_genai.models | AFC is enabled with max remote calls: 10.
2026-04-27 15:11:17 | INFO     | httpx | HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent "HTTP/1.1 200 OK"
2026-04-27 15:11:17 | INFO     | llm_judge | Evaluation complete | label='EPISTEMIC' | latency=1.771s


...

[5/7] id=medqa_11 label=EPISTEMIC error=None


2026-04-27 15:11:24 | INFO     | llm_judge | Evaluating text | provider=GoogleGenAI | text_preview='Has he had any prior immunological workup, specifically immu...'
2026-04-27 15:11:24 | INFO     | google_genai.models | AFC is enabled with max remote calls: 10.
2026-04-27 15:11:26 | INFO     | httpx | HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent "HTTP/1.1 200 OK"
2026-04-27 15:11:26 | INFO     | llm_judge | Evaluation complete | label='EPISTEMIC' | latency=2.263s


[6/7] id=medqa_9 label=EPISTEMIC error=None


2026-04-27 15:11:27 | INFO     | llm_judge | Evaluating text | provider=GoogleGenAI | text_preview='Is the cough associated with any shortness of breath, especi...'
2026-04-27 15:11:27 | INFO     | google_genai.models | AFC is enabled with max remote calls: 10.
2026-04-27 15:11:29 | INFO     | httpx | HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent "HTTP/1.1 200 OK"
2026-04-27 15:11:29 | INFO     | llm_judge | Evaluation complete | label='EPISTEMIC' | latency=1.839s


[7/7] id=medqa_6 label=EPISTEMIC error=None
Saved classified results to: D:\final_project\pilot_study_2\LLM Judge\phase1_cq_type_classified.csv
Rows saved: 7


## 8) Output Validation and Quick Summary

In [12]:
required_cols = [
    "id",
    "clarifying_question",
    "cq_type",
    "judge_raw_response",
    "judge_error",
]
missing_cols = [c for c in required_cols if c not in classified_df.columns]
if missing_cols:
    raise RuntimeError(f"Missing required output columns: {missing_cols}")

invalid_labels_df = classified_df[~classified_df["cq_type"].isin(list(VALID_LABELS))]
error_df = classified_df[classified_df["judge_error"].notna()]

print(f"Total classified rows: {len(classified_df)}")
print("Label distribution:")
print(classified_df["cq_type"].value_counts(dropna=False))
print()
print(f"Rows with invalid labels: {len(invalid_labels_df)}")
print(f"Rows with judge errors: {len(error_df)}")

if len(invalid_labels_df) > 0:
    display(invalid_labels_df[["id", "clarifying_question", "cq_type", "judge_raw_response"]].head(10))

display(classified_df[["id", "clarifying_question", "cq_type", "judge_latency_seconds"]].head(10))

Total classified rows: 7
Label distribution:
cq_type
EPISTEMIC    7
Name: count, dtype: int64

Rows with invalid labels: 0
Rows with judge errors: 0


,id,clarifying_question,cq_type,judge_latency_seconds
0,medqa_3,"Are you experiencing any fever, chills, or spe...",EPISTEMIC,2.997
1,medqa_1,"Are the blisters itchy, painful, or burning?",EPISTEMIC,3.499
2,medqa_7,Does he experience shortness of breath when ly...,EPISTEMIC,1.771
3,medqa_4,Has your difficulty breathing worsened when ly...,EPISTEMIC,2.087
4,medqa_11,Has she experienced any uterine contractions o...,EPISTEMIC,2.352
5,medqa_9,"Has he had any prior immunological workup, spe...",EPISTEMIC,2.263
6,medqa_6,Is the cough associated with any shortness of ...,EPISTEMIC,1.839
